# Train the pothole detector on Colab (YOLOv8, single class)

Runs `scripts/train_pothole.py` from the project ZIP. Training is the only stage that needs a Colab GPU.
Switch the runtime to **GPU** first (Runtime > Change runtime type).

Before you start, put two files in Google Drive (folder `MyDrive/ptr/`):
1. the project ZIP (`pothole-mtw-risk.zip`),
2. a zip of `data/processed/pothole` (it must contain `train/`, `val/`, `test_a/`; create it locally with
   `cd data/processed && zip -r pothole_dataset.zip pothole`).

Nothing here is a result: metrics are whatever training prints. This notebook never invents numbers.

In [ ]:
# ---- settings (edit here) ----------------------------------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/ptr"
REPO_ZIP     = f"{DRIVE_ROOT}/pothole-mtw-risk.zip"
DATASET_ZIP  = f"{DRIVE_ROOT}/pothole_dataset.zip"
RUNS_DIR     = f"{DRIVE_ROOT}/runs"          # training writes straight to Drive: last.pt survives disconnects
RUN_NAME     = "yolov8s_640"                 # change it for every new experiment
MODEL        = "yolov8s.pt"                  # COCO-pretrained start (yolov8n.pt for the speed comparison)
IMGSZ, EPOCHS, PATIENCE, BATCH = 640, 100, 20, 16
ULTRALYTICS_VERSION = "8.4.212"              # version tested by the repo author; use the same one locally

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Copy the project and the dataset to the LOCAL disk (Drive reads are slow)

In [ ]:
import os, pathlib, shutil, zipfile

def fetch(zip_on_drive, dest):
    local = f"/content/{os.path.basename(zip_on_drive)}"
    shutil.copy(zip_on_drive, local)                   # copy first, then unzip on the fast local disk
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(local) as z:
        z.extractall(dest)
    os.remove(local)

fetch(REPO_ZIP, "/content/work")
fetch(DATASET_ZIP, "/content/dataset")

REPO = next(p for p in pathlib.Path("/content/work").rglob("train_pothole.py")).parent.parent
DATA_ROOT = next(p.parent.parent for p in pathlib.Path("/content/dataset").rglob("images") if p.parent.name == "train")
print("repo:", REPO)
print("dataset root:", DATA_ROOT, [d.name for d in DATA_ROOT.iterdir()])

In [ ]:
# Dataset yaml with ABSOLUTE local paths (the yaml in the repo points at your laptop)
import yaml
ds = {"path": str(DATA_ROOT), "train": "train/images", "val": "val/images", "test": "test_a/images",
      "names": {0: "pothole"}}
pathlib.Path("/content/dataset_colab.yaml").write_text(yaml.safe_dump(ds, sort_keys=False))
for s in ("train", "val", "test_a"):
    n = len(list((DATA_ROOT / s / "images").glob("*")))
    print(s, n, "images")

## 3. Install only the direct dependencies (leave Colab's PyTorch alone)

In [ ]:
from importlib.metadata import version
torch_before = version("torch")
%pip install -q "ultralytics=={ULTRALYTICS_VERSION}" pyyaml
torch_after = version("torch")
print("torch before:", torch_before, "| after:", torch_after)
assert torch_before == torch_after, "pip changed Colab's torch: restart the runtime and do not reinstall torch"
import torch; print("CUDA available:", torch.cuda.is_available())

## 4. Train (or resume after a disconnect: just run this cell again)

In [ ]:
last = pathlib.Path(RUNS_DIR) / RUN_NAME / "weights" / "last.pt"
flag = "--resume" if last.exists() else ""
print("resuming from", last if flag else "(new run)")
!cd {REPO} && PYTHONPATH=src python scripts/train_pothole.py --config configs/train_pothole.yaml \
    --data /content/dataset_colab.yaml --model {MODEL} --imgsz {IMGSZ} --epochs {EPOCHS} \
    --patience {PATIENCE} --batch {BATCH} --project {RUNS_DIR} --name {RUN_NAME} {flag}

If the session disconnects, reconnect, re-run cells 1-3 and this cell: `last.pt` on Drive is picked up and Ultralytics
resumes from the last saved epoch (resume keeps the original arguments). A finished run reports
"nothing to resume": use a new `RUN_NAME` for the next experiment.

## 5. Save the weights (with checksum) and look at the training log

In [ ]:
import hashlib, pandas as pd
run = pathlib.Path(RUNS_DIR) / RUN_NAME
best = run / "weights" / "best.pt"
out = pathlib.Path(DRIVE_ROOT) / "models"; out.mkdir(exist_ok=True)
shutil.copy(best, out / f"{RUN_NAME}_best.pt")
sha = hashlib.sha256(best.read_bytes()).hexdigest()
(out / f"{RUN_NAME}_best.pt.sha256").write_text(sha + "\n")
print("saved", out / f"{RUN_NAME}_best.pt", "\nsha256", sha)
print(pd.read_csv(run / "results.csv").tail(5).to_string())   # whatever training logged; copy it to docs/training_log_template.md

## 6. Next (on your own machine)
Download `..._best.pt` from Drive, save it as `models/pothole/best.pt`, record the sha256 and the settings in
`models/pothole/model_card_template.md`, then run `scripts/evaluate_detection.py` on val, Test A and Test B.